# E7: Online Learning with Observation-Centric Memory

## Improvements over E5

| # | Improvement | Problem it solves |
|---|------------|------------------|
| 1 | **Observation-centric context** | Expert says "go to countertop 1" but object is on diningtable → show expert's *situation* so LLM adapts |
| 2 | **Strategy templates** | Abstract plan ("find → take → cool → place") before specific actions |
| 3 | **Online reward annotation** | Agent learns from own experience — wins=1.0, fails=0.0 |
| 4 | **Expert reward decay** | Experts retrieved in failed games get 10% penalty → noisy experts filtered out |

## The Transfer Problem

Expert actions are correct in their game but may not transfer to a different room layout:
```
Expert game:  "On countertop 1, you see tomato 1" → "take tomato 1 from countertop 1"
Eval game:    "On diningtable 3, you see tomato 2" → agent needs "take tomato 2 from diningtable 3"
```

**Fix**: Show the expert's OBSERVATION alongside their action. The LLM can then:
1. See what the expert saw ("countertop 1 had the tomato")  
2. See what the expert did ("took the tomato from there")
3. Adapt to current layout ("I see tomato 2 on diningtable 3 → take tomato 2 from diningtable 3")

## Online Learning Protocol

```
Round 1: Play 30 games with expert memory → annotate wins/fails
Round 2: Play 30 games with expert + own experience (reward-filtered)
Round 3: Play 30 games with refined memory (noisy experts decayed)
```

**Model**: Ollama qwen2.5:7b (zero cost)


In [ ]:
"""
E7: Online Learning with CVX — Ollama prototype

Improvements over E5:
1. Observation-centric retrieval: show expert's SITUATION + action (not just action)
2. Task-type strategy templates (abstract plan before specific actions)
3. Online reward annotation (learn from own experience across rounds)
4. Expert reward decay (penalize experts that lead to failures)
5. All on Ollama (qwen2.5:7b) — zero API cost
"""
import os, json, time
import numpy as np
from sentence_transformers import SentenceTransformer
from openai import OpenAI

# === CONFIG: OLLAMA ===
OLLAMA_URL = 'http://hpc.glaciar.lab:11434/v1'
client = OpenAI(base_url=OLLAMA_URL, api_key='ollama')
LLM_MODEL = 'qwen2.5-coder:7b-instruct'

MAX_STEPS = 30
N_EVAL = 30
N_ROUNDS = 3
TOP_K = 3

embedder = SentenceTransformer('all-MiniLM-L6-v2')
print(f'LLM: {LLM_MODEL} (Ollama)')

import chronos_vector as cvx

# === TASK TYPE DETECTION ===
TASK_TYPES = {
    'heat': 'heat_then_place',
    'cool': 'cool_then_place',
    'clean': 'clean_then_place',
    'examine': 'examine_in_light',
    'look at': 'examine_in_light',
    'find two': 'pick_two',
    'put two': 'pick_two',
}

def detect_task_type(task_text):
    for keyword, ttype in TASK_TYPES.items():
        if keyword in task_text.lower():
            return ttype
    return 'pick_and_place'

STRATEGY_TEMPLATES = {
    'pick_and_place': '1) Search locations for the object 2) Take it 3) Go to target 4) Put it',
    'heat_then_place': '1) Find object 2) Take it 3) Go to microwave 4) Heat it 5) Take from microwave 6) Go to target 7) Put it',
    'cool_then_place': '1) Find object 2) Take it 3) Go to fridge 4) Cool it 5) Take from fridge 6) Go to target 7) Put it',
    'clean_then_place': '1) Find object 2) Take it 3) Go to sinkbasin 4) Clean it 5) Go to target 6) Put it',
    'examine_in_light': '1) Find object 2) Take it 3) Find a lamp (desklamp/floorlamp) 4) Use lamp',
    'pick_two': '1) Find first object 2) Take it 3) Go to target 4) Put it 5) Find second object 6) Take it 7) Go to target 8) Put it',
}

# === BUILD INDEX WITH OBSERVATIONS AS METADATA ===
with open('data/episodic/alfworld_metadata.json') as f:
    meta = json.load(f)

print('Building index with observation metadata...')
index = cvx.TemporalIndex(m=16, ef_construction=100)

# Store step data for context formatting
# step_data[timestamp] = {"action": ..., "observation": ..., "task": ...}
step_data = {}

for ep_id_str, ep in meta.items():
    ep_id = int(ep_id_str)
    task = ep.get('task', '')
    steps = ep.get('steps', [])

    for si, step in enumerate(steps):
        action = step.get('action', '')
        observation = step.get('observation', '')
        text = f"{task} | {action} | {observation}"
        vec = embedder.encode(text).tolist()

        entity_id = ep_id
        timestamp = ep_id * 10000 + si

        index.insert(entity_id, timestamp, vec, reward=1.0)

        step_data[timestamp] = {
            'action': action,
            'observation': observation[:200],
            'task': task,
        }

print(f'Index: {len(index)} points, {len(step_data)} step records')

# === CONTEXT FORMATTING: Show expert SITUATION + action ===
def format_context(results, task_type):
    """Show what the expert SAW and DID — not just the action."""
    if not results:
        return ''

    strategy = STRATEGY_TEMPLATES.get(task_type, '')
    parts = []
    if strategy:
        parts.append(f'General strategy: {strategy}\n')

    parts.append('What expert agents did in similar situations:')
    for i, r in enumerate(results[:TOP_K]):
        ep_id = r['entity_id']
        # Show the matched step's observation
        match_ts = None
        # Find the timestamp of the match (entity_id's trajectory, closest to match)
        traj = index.trajectory(ep_id)
        if traj:
            # The match node — get its data
            # We need to find which timestamp corresponds to the match
            # The match is the node_id, but we stored entity_id = ep_id
            # We can use the score to identify, but simpler: show successors
            pass

        successors = r.get('successors', [])
        if not successors:
            continue

        parts.append(f'\n  Expert {i+1}:')
        for j, (nid, ts, vec) in enumerate(successors[:4]):
            sd = step_data.get(ts, {})
            action = sd.get('action', '?')
            obs = sd.get('observation', '')
            if action != '?':
                if obs and j == 0:
                    # Show observation for first successor to give context
                    parts.append(f'    Saw: "{obs[:120]}"')
                parts.append(f'    Did: {action}')

    return '\n'.join(parts) + '\n'

# === AGENT ===
def llm_call(system, user):
    try:
        resp = client.chat.completions.create(
            model=LLM_MODEL,
            messages=[{'role': 'system', 'content': system},
                      {'role': 'user', 'content': user}],
            temperature=0.0, max_tokens=100,
        )
        return resp.choices[0].message.content.strip()
    except:
        return ''

def agent_act(observation, admissible, task, history, context=''):
    system = ('You are a household agent. Choose ONE action from the list. '
              'Output ONLY the action text, nothing else.')
    user = f'Task: {task}\n\n'
    if context:
        user += context + '\n'
    if history:
        user += 'Your recent actions:\n'
        for h in history[-5:]:
            user += f'  > {h["a"]} -> {h["o"][:60]}\n'
        user += '\n'
    user += f'Current observation: {observation}\n\nAvailable actions:\n'
    for a in admissible:
        user += f'  - {a}\n'
    user += '\nChoose ONE action:'

    chosen = llm_call(system, user).lower().strip()
    # Match to admissible
    for a in admissible:
        if a.lower() == chosen:
            return a
    for a in admissible:
        if a.lower() in chosen or chosen in a.lower():
            return a
    return admissible[0]

def run_game(env, index, use_memory=False):
    obs, info = env.reset()
    observation = obs[0]
    task = (observation.split('Your task is to:')[-1].strip().split('\n')[0]
            if 'Your task is to:' in observation else '')
    task_type = detect_task_type(task)
    history = []
    used_retrievals = []

    for step in range(MAX_STEPS):
        admissible = info['admissible_commands'][0]
        ctx = ''
        if use_memory:
            try:
                qv = embedder.encode(f'{task} | {observation}').tolist()
                results = index.causal_search(vector=qv, k=TOP_K, temporal_context=5)
                ctx = format_context(results, task_type)
                used_retrievals.extend(results)
            except:
                pass

        action = agent_act(observation, admissible, task, history, ctx)
        obs, rewards, dones, info = env.step([action])
        observation = obs[0]
        history.append({'a': action, 'o': observation[:150]})
        if dones[0]:
            break

    won = info['won'][0]
    return {
        'task': task, 'task_type': task_type, 'won': won,
        'steps': len(history), 'history': history,
        'used_retrievals': used_retrievals,
    }

# === ALFWORLD ===
from alfworld.agents.environment.alfred_tw_env import AlfredTWEnv
config = {
    'dataset': {
        'data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/train'),
        'eval_id_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_seen'),
        'eval_ood_data_path': os.path.expanduser('~/.cache/alfworld/json_2.1.1/valid_unseen'),
        'num_train_games': 0, 'num_eval_games': 0,
    },
    'env': {'goal_desc_human_anns_prob': 0.0, 'task_types': [1, 2, 3, 4, 5, 6],
            'domain_randomization': False, 'expert_type': 'handcoded'},
    'general': {'training_method': 'dqn', 'random_seed': 42},
    'rl': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'dagger': {'training': {'max_nb_steps_per_episode': MAX_STEPS}},
    'logic': {'domain': os.path.expanduser('~/.cache/alfworld/logic/alfred.pddl'),
              'grammar': os.path.expanduser('~/.cache/alfworld/logic/alfred.twl2')},
}
env_wrapper = AlfredTWEnv(config, train_eval='eval_out_of_distribution')
env = env_wrapper.init_env(batch_size=1)
print(f'ALFWorld: {len(env_wrapper.game_files)} games\n')

# === ONLINE LEARNING LOOP ===
all_round_results = {}
next_ep_id = 1000

for round_num in range(N_ROUNDS):
    print(f'\n{"#"*60}')
    print(f'ROUND {round_num + 1}/{N_ROUNDS} — Index: {len(index)} points')
    print(f'{"#"*60}')

    results = []
    wins = 0
    for g in range(N_EVAL):
        r = run_game(env, index, use_memory=True)
        results.append(r)
        if r['won']:
            wins += 1
        print(f'  {g+1}/{N_EVAL}: {"WIN" if r["won"] else "fail"} ({r["steps"]}s) '
              f'{r["task_type"][:15]} [{wins}/{g+1}={wins/(g+1)*100:.0f}%]', flush=True)

    rate = wins / N_EVAL * 100
    print(f'\n  >>> Round {round_num + 1}: {wins}/{N_EVAL} = {rate:.1f}%')

    cond = f'Round{round_num + 1}'
    all_round_results[cond] = [
        {'won': r['won'], 'steps': r['steps'], 'task': r['task'], 'task_type': r['task_type']}
        for r in results
    ]

    # --- Online learning: add own experience ---
    added = 0
    for r in results:
        reward = 1.0 if r['won'] else 0.0
        for si, step in enumerate(r['history']):
            text = f"{r['task']} | {step['a']} | {step['o']}"
            vec = embedder.encode(text).tolist()
            ts = next_ep_id * 10000 + si
            index.insert(next_ep_id, ts, vec, reward=reward)
            step_data[ts] = {
                'action': step['a'],
                'observation': step['o'][:200],
                'task': r['task'],
            }
            added += 1
        next_ep_id += 1

    # --- Reward decay: penalize experts retrieved in failed games ---
    decayed = 0
    for r in results:
        if not r['won']:
            for ret in r.get('used_retrievals', []):
                for nid, ts, vec in ret.get('successors', []):
                    cur = index.reward(nid)
                    if cur is not None:
                        index.set_reward(nid, max(0.0, cur * 0.9))
                        decayed += 1

    print(f'  Online learning: +{added} own steps, decayed {decayed} expert rewards')

    # Save incrementally
    with open('data/episodic/e7_online_results.json', 'w') as f:
        json.dump(all_round_results, f, indent=2)

# === SUMMARY ===
print(f'\n{"="*60}')
print('ONLINE LEARNING RESULTS')
print(f'{"="*60}')
for cond, results in all_round_results.items():
    wins = sum(1 for r in results if r['won'])
    print(f'  {cond}: {wins}/{len(results)} = {wins/len(results)*100:.1f}%')

# Per task type (last round)
print(f'\nPer task type (last round):')
last = list(all_round_results.values())[-1]
by_type = {}
for r in last:
    tt = r.get('task_type', '?')
    by_type.setdefault(tt, []).append(r['won'])
for tt, outcomes in sorted(by_type.items()):
    w = sum(outcomes)
    print(f'  {tt}: {w}/{len(outcomes)} = {w/len(outcomes)*100:.0f}%')

print('\nSaved to data/episodic/e7_online_results.json')
